# 00 · División de Wildfire Smoke por cámara

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elianpinzas/vision_por_computadora_2_tp_final/blob/gonzalo-yolov8/notebooks/00_division_eventos.ipynb)

**Responsable:** Gonzalo Rodríguez · **Código:** `src/division.py` · **Parámetros:** sección `division` de `config.yaml`

**Objetivo:** reemplazar la división aleatoria de Roboflow (516/147/74) por una división **por cámara**, para que no haya imágenes iguales o casi iguales entre train y valid/test.

**Por qué:** las imágenes son frames de cámaras fijas de HPWREN. La división aleatoria dejó frames del mismo incendio, tomados con minutos de diferencia por la misma cámara, repartidos entre train y test. Eso infla las métricas: el modelo se evalúa con imágenes que prácticamente ya vio.

**Método:**
1. Se mide la fuga de la división original con *dHash* (huella de 256 bits por imagen) y varios umbrales.
2. Se agrupan las imágenes por **vista de cámara**, comparando el paisaje (mitad inferior, sin el cielo).
3. Se revisan los grupos a ojo y, si hace falta, se corrigen.
4. Se reparten **cámaras completas** en ~70/20/10 de imágenes.
5. Se verifica que no queden gemelas entre splits y se exporta el CSV con la división.

*Nota:* se intentó leer la fecha y la cámara impresas en cada imagen con OCR (tesseract), pero el texto llega a medir ~7 px y no se lee de forma confiable. Por eso se agrupa por similitud visual.

## 0. Preparación del entorno

In [ ]:
import os
RAMA = "gonzalo-yolov8"
if not os.path.exists("vision_por_computadora_2_tp_final"):
    !git clone -b {RAMA} https://github.com/elianpinzas/vision_por_computadora_2_tp_final.git
%cd vision_por_computadora_2_tp_final
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.utils import load_config, set_seed
from src import datos, division as dv

cfg = load_config("config.yaml")
SEED = cfg["seed"]
set_seed(SEED)
D = cfg["division"]          # parámetros de esta división
FIG_DIR = Path("results/division"); FIG_DIR.mkdir(parents=True, exist_ok=True)
D

## 1. Descargar el dataset (versión fija en Roboflow)
Necesita el secret `ROBOFLOW_API_KEY` y `workspace` / `project` / `version` completos en `config.yaml`.

In [ ]:
DATA_DIR = Path(datos.descargar_wildfire(cfg))
for s in dv.SPLITS:
    print(s, len(list((DATA_DIR / s / "images").glob("*"))), "imágenes")

## 2. Inventario: una fila por imagen

In [ ]:
df = dv.inventario(DATA_DIR)
print(len(df), "imágenes ·", df.n_cajas.sum(), "cajas ·", df.escala_grises.sum(), "en escala de grises")
print("Tamaños:", df.groupby(["ancho", "alto"]).size().to_dict())
df.groupby("split_original").agg(imagenes=("imagen", "size"), cajas=("n_cajas", "sum")).reindex(dv.SPLITS)

## 3. Fuga en la división original
**dHash:** cada imagen se reduce a 16×17 en grises y se codifica en 256 bits (si cada píxel es más claro que su vecino). La **distancia de Hamming** cuenta en cuántos bits difieren dos imágenes: 0 = idénticas. Se considera "gemela" a una imagen de valid/test que tiene en train otra a distancia ≤ `umbral_gemela`.

Como el resultado depende del umbral, se informa para varios valores.

In [ ]:
DIST = dv.matriz_distancias(df)
sens_orig = dv.sensibilidad_umbral(df, DIST, "split_original", D["umbrales_sensibilidad"])
g, n = dv.contar_gemelas(df, DIST, "split_original", D["umbral_gemela"])
print(f"Umbral elegido ({D['umbral_gemela']}): {g} de {n} imágenes de valid+test tienen una casi gemela en train")
sens_orig

In [ ]:
dmin = dv.distancia_minima_a_train(df, DIST, "split_original")
plt.figure(figsize=(6, 3.2))
plt.hist(dmin, bins=range(0, int(dmin.max()) + 2), color="#888", edgecolor="white")
plt.axvline(D["umbral_gemela"] + 0.5, color="red", ls="--", label=f"umbral = {D['umbral_gemela']}")
plt.xlabel("distancia dHash al vecino más cercano en train"); plt.ylabel("imágenes de valid+test")
plt.title("División original de Roboflow"); plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / "distancia_a_train_original.png", dpi=110); plt.show()
print("Mediana de la distancia:", np.median(dmin))

## 4. Agrupar por vista de cámara
Descriptor: mitad inferior de la imagen (paisaje), en grises, 32×16, normalizada. Se unen las imágenes con similitud coseno > `umbral_camara`. Se muestra cuántos grupos salen con umbrales cercanos para comprobar que el resultado es estable.

In [ ]:
for u in (D["umbral_camara"] - 0.05, D["umbral_camara"], D["umbral_camara"] + 0.03):
    print(f"umbral {u:.2f}: {len(set(dv.agrupar_por_camara(df, u)))} grupos")
df["grupo"] = dv.agrupar_por_camara(df, D["umbral_camara"])
df.groupby("grupo").agg(imagenes=("imagen", "size"), cajas=("n_cajas", "sum"),
                        grises=("escala_grises", "sum")).T

## 5. Revisión visual de los grupos
Cada fila tiene que ser **una sola cámara**, y dos filas no deberían ser la misma cámara. Si algo está mal, se corrige en la celda siguiente.

In [ ]:
rng = np.random.default_rng(SEED)
grupos = sorted(df.grupo.unique())
fig, ejes = plt.subplots(len(grupos), 5, figsize=(12, 1.9 * len(grupos)))
for fila, g in zip(ejes, grupos):
    rutas = df[df.grupo == g].ruta.values
    elegidas = rng.choice(rutas, size=min(5, len(rutas)), replace=False)
    for k, ax in enumerate(fila):
        ax.axis("off")
        if k < len(elegidas):
            ax.imshow(Image.open(elegidas[k]))
    fila[0].set_title(f"grupo {g} · {len(rutas)} imágenes", fontsize=8, loc="left")
plt.tight_layout(); plt.savefig(FIG_DIR / "grupos_camara.jpg", dpi=80); plt.show()

In [ ]:
# Correcciones manuales (dejar vacías si los grupos están bien)
UNIR = []    # ej.: [(3, 17)] -> el grupo 17 pasa a ser parte del 3
MOVER = {}   # ej.: {"<clave de la imagen>": 5} -> esa imagen pasa al grupo 5
for destino, origen in UNIR:
    df.loc[df.grupo == origen, "grupo"] = destino
for k, g in MOVER.items():
    df.loc[df.clave == k, "grupo"] = g

## 6. Repartir cámaras completas en ~70/20/10
Se prueban `intentos` asignaciones al azar (con la semilla del grupo) y se elige la que deja imágenes y cajas más cerca del objetivo, con al menos `min_camaras_val_test` cámaras en valid y en test.

In [ ]:
grupo_a_split = dv.repartir_grupos(df, D["objetivo"], D["min_camaras_val_test"], D["intentos"], SEED)
df["split_camara"] = df.grupo.map(grupo_a_split)
print({s: sorted(g for g, v in grupo_a_split.items() if v == s) for s in dv.SPLITS})
dv.resumen_division(df, "split_camara")

## 7. Verificación

In [ ]:
sens_nueva = dv.sensibilidad_umbral(df, DIST, "split_camara", D["umbrales_sensibilidad"])
comparacion = sens_orig[["umbral", "gemelas"]].rename(columns={"gemelas": "original"}) \
    .assign(por_camara=sens_nueva.gemelas.values, valid_test=sens_orig["valid+test"].values)
print("Imágenes de valid+test con una casi gemela en train, según el umbral:")
display(comparacion)

g_new, _ = dv.contar_gemelas(df, DIST, "split_camara", D["umbral_gemela"])
assert g_new == 0, "Quedan gemelas con el umbral elegido: revisar los grupos (sección 5)"
assert df.groupby("grupo").split_camara.nunique().max() == 1, "Hay una cámara repartida en dos splits"
print("OK: ninguna gemela entre splits y cada cámara está en un solo split")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
pd.crosstab(df.split_original, df.split_camara).reindex(index=dv.SPLITS, columns=dv.SPLITS).plot.bar(ax=ax[0])
ax[0].set_title("Imágenes: split original vs. split por cámara"); ax[0].set_xlabel("split original")
comparacion.set_index("umbral")[["original", "por_camara"]].plot.bar(ax=ax[1])
ax[1].set_title("Gemelas en train según el umbral"); ax[1].set_ylabel("imágenes de valid+test")
plt.tight_layout(); plt.savefig(FIG_DIR / "division_resumen.png", dpi=110); plt.show()

## 8. Exportar el CSV de la división

In [ ]:
CSV = dv.guardar_csv(df, D["csv"])
print("Guardado:", CSV, "·", len(df), "filas")

## 9. Armar la carpeta del dataset con la nueva división
`dv.aplicar_division` copia imágenes y etiquetas de la descarga según el CSV y escribe su `data.yaml`. Los notebooks de entrenamiento pueden usar el atajo `dv.preparar_wildfire_por_camara(cfg, DATA_DIR)`.

In [ ]:
DATA_YAML = dv.aplicar_division(CSV, DATA_DIR, D["local_dir"])
for s in dv.SPLITS:
    print(s, len(list((Path(D["local_dir"]) / s / "images").glob("*"))))
print(DATA_YAML.read_text())

## 10. Guardar en el repo
Subir a la rama `gonzalo-yolov8`: el CSV y las figuras de `results/division/` (descargarlos acá y subirlos con git desde la compu).

In [ ]:
from google.colab import files
for f in [CSV, *sorted(FIG_DIR.glob("*"))]:
    files.download(str(f))